# Part 4 — Zero-Shot Robust Loss Evaluation (Colab GPU)

This notebook runs your final Part 4 robust-loss benchmark on medical vision data.

## Datasets
- PathMNIST
- DermaMNIST

## Models (non-generative)
- `google/medsiglip-448`
- `openai/clip-vit-base-patch32`
- `vinid/plip`
- `microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224`

## Experiment batteries
- A: clean labels
- B: uniform noise
- C: class-dependent noise
- D: GCE q-sweep


In [ ]:
# ── Cell 1: Install pinned dependencies ───────────────────────────────────────
!pip install -q --upgrade \
    typing_extensions>=4.10.0 \
    torch>=2.6.0 \
    torchvision>=0.21.0 \
    transformers>=4.46.0 \
    datasets>=2.20.0 \
    accelerate>=0.33.0 \
    huggingface_hub>=0.24.0 \
    medmnist>=3.0.2 \
    open_clip_torch>=2.24.0 \
    ftfy>=6.2.0 \
    regex>=2024.0.0 \
    scikit-learn>=1.3.0 \
    matplotlib>=3.8.0 \
    seaborn>=0.13.0 \
    pandas>=2.1.0 \
    pillow>=10.0.0 \
    tqdm>=4.66.0

print('✓ Dependencies installed (pinned).')
print('If imports fail in same session: Runtime -> Restart session, then continue.')


In [ ]:
# ── Cell 2: Runtime sanity checks ─────────────────────────────────────────────
import importlib.metadata as md

def ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return 'not-installed'

print('typing_extensions:', ver('typing_extensions'))
print('torch:', ver('torch'))
print('transformers:', ver('transformers'))
print('medmnist:', ver('medmnist'))
print('open_clip_torch:', ver('open_clip_torch'))

import torch
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f'✓ GPU : {dev.name}  VRAM = {dev.total_memory / 1024**3:.1f} GB')
    print(f'  CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}')
else:
    print('⚠ No GPU — Runtime → Change runtime type → GPU')


In [ ]:
# ── Cell 3: Upload script ────────────────────────────────────────────────────
import os
SCRIPT = 'part4_Multimodal_Vision_Robust_Experiments.py'

if not os.path.exists(SCRIPT):
    from google.colab import files
    print(f'Please upload {SCRIPT}')
    uploaded = files.upload()
    assert SCRIPT in uploaded, f'Expected {SCRIPT}, got {list(uploaded.keys())}'

print(f'✓ {SCRIPT} ready ({os.path.getsize(SCRIPT)/1024:.0f} KB)')

In [ ]:
# ── Cell 4: (Optional) Set HF token ──────────────────────────────────────────
# Uncomment ONLY if needed:
# from google.colab import userdata
# import os
# os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

In [ ]:
# ── Cell 5: RUN robust-loss experiment pipeline ───────────────────────────────
# Optional controls:
#   %env ROBUST_NN_MODEL_ONLY=MedSigLIP
#   %env ROBUST_NN_MAX_SAMPLES=1200
#   %env ROBUST_NN_QUICK_RUN=1

import runpy
runpy.run_path(SCRIPT, run_name='__main__')


In [ ]:
# ── Cell 6: List results ─────────────────────────────────────────────────────
from pathlib import Path
rd = Path('results_multimodal_vision')
if rd.exists():
    for f in sorted(rd.iterdir()):
        print(f'  {f.name:45s}  {f.stat().st_size/1024:7.1f} KB')
else:
    print('Results folder not yet created — run Cell 5 first.')

In [ ]:
# ── Cell 7: Preview summary table ────────────────────────────────────────────
import pandas as pd
from pathlib import Path
sp = Path('results_multimodal_vision') / 'summary_all.csv'
if sp.exists():
    df = pd.read_csv(sp)
    cols = [
        'dataset', 'model_key', 'battery', 'noise_type', 'noise_rate',
        'loss', 'loss_value', 'acc_clean_labels', 'acc_noisy_labels'
    ]
    cols = [c for c in cols if c in df.columns]
    print(df[cols].to_string(index=False))
else:
    print('Not yet generated — run Cell 5 first.')


In [ ]:
# ── Cell 8: Download results ─────────────────────────────────────────────────
import shutil
from google.colab import files
rd = Path('results_multimodal_vision')
if rd.exists():
    arc = shutil.make_archive('results_multimodal_vision', 'zip', '.', 'results_multimodal_vision')
    files.download(arc)
else:
    print('No results to download yet.')